<a href="https://colab.research.google.com/github/xwang335/Campbell-A/blob/main/pull_rawdata.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [3]:
# %%
import argparse
import pandas as pd
import numpy as np
import os
import wrds
import warnings
warnings.filterwarnings('ignore')

from google.colab import drive
drive.mount('/content/drive')

print('Setup complete.')

path = "/content/drive/MyDrive/Campbell A data/"

os.chdir(path)

Mounted at /content/drive
Setup complete.


In [6]:
# %%
def pull_data_from_wrds(start, end, freq):
    # Connect to WRDS
    db = wrds.Connection()

    # Pull data from the 'msf' table for the specified date range
    if freq == 'M':
        query = f"""
        SELECT
            b.permno,
            b.date,
            b.ret,
            b.prc,
            b.shrout,
            c.rf,
            (b.ret - c.rf) AS exret
        FROM crsp.msf b
        LEFT JOIN ff.factors_monthly c
            ON date_trunc('month', b.date) = date_trunc('month', c.date)
        WHERE b.date >= '{start}'
        AND b.date <= '{end}'
        AND b.ret IS NOT NULL
        """
    if freq == 'D':
        query = f"""
        select
            permno,
            date,
            ret,
            prc,
            vol,
            shrout,
            askhi,
            bidlo
        from crsp.dsf
        WHERE date >= '{start}'
        AND date <= '{end}'
        """
    df = db.raw_sql(query)
    db.close()
    df['me'] = df['prc'].abs() * df['shrout']
    return df

# %%
def clean_data(df):
    # Remove rows with missing values in 'ret' column
    df['date'] = pd.to_datetime(df['date'])
    df = df.sort_values(['permno', 'date']).reset_index(drop=True)
    print(df.describe())  # Debugging line to check data after cleaning
    return df


In [10]:
start='1956-01-01'
end='2016-12-31'

if not os.path.exists("crsp_monthly.parquet"):
        df = pull_data_from_wrds(start=start, end=end, freq='M')
        df = clean_data(df)
        print(df['exret'])
        df.to_parquet("crsp_monthly.parquet")
